# 02 — Training-only ECG and CXR FAISS banks

Build separate training-only normalized inner-product banks. Every query excludes its own patient. Embeddings are normalized once; FAISS and float64 cosine scores are compared in batches. Top-50 pools retain score/record-ID ordering, with scalar recomputation around the selection boundary. Same-modality retrieval does not require a shared ECG–CXR embedding space.

**Resume:** completed batches are atomically saved under `OUT/bank_checkpoints`. After a job interruption, rerun this notebook with the same inputs and software. Only the interrupted batch needs repeating; input loading and index construction are repeated. Changed inputs/code/library versions are rejected. Old versions had no query checkpoints, so their unfinished query work cannot be recovered. Progress streams to this notebook and `OUT/build_banks.log`. Notebook 02 builds banks; model training occurs in 03.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
assert (ROOT / 'core.py').exists(), 'Start the notebook kernel in graph_reasoning/'
sys.path.insert(0, str(ROOT))
from core import load, audit
CONFIG = load(ROOT / 'config.json')
OUT = ROOT / CONFIG['output_dir']
OUT.mkdir(parents=True, exist_ok=True)


In [ ]:
import subprocess
import json

print('Python interpreter:', sys.executable)
print('Input/output directory:', OUT.resolve())
script = ROOT / 'build_banks.py'
if not script.is_file():
    raise FileNotFoundError(f'Upload build_banks.py to {ROOT.resolve()} before running notebook 02.')
required = [OUT / name for name in ('records.json', 'targets.json', 'audit.json')]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError(
        'Notebook 01 has not produced the required files in this output directory:\n'
        + '\n'.join(missing)
        + '\nRun notebook 01 until it prints READY. Notebook 00b alone is not sufficient.'
    )
if CONFIG.get('mode') not in ('real', 'synthetic'):
    raise ValueError('Invalid mode in config.json')
expected_synthetic = CONFIG['mode'] == 'synthetic'
rows = load(OUT / 'records.json')
targets = load(OUT / 'targets.json')
verified = audit(rows, targets, synthetic=expected_synthetic)
if load(OUT / 'audit.json') != json.loads(json.dumps(verified)):
    raise ValueError('Saved audit does not match the current inputs. Rerun notebook 01.')
# Real source exports must still match the copies produced by notebook 01.
if not expected_synthetic:
    for key, filename in [('records_path', 'records.json'), ('targets_path', 'targets.json')]:
        source = Path(CONFIG[key]).expanduser()
        if not source.is_absolute(): source = ROOT / source
        if not source.is_file() or load(source) != load(OUT / filename):
            raise ValueError(f'{key} is missing or changed since notebook 01. Rerun notebook 01.')
for modality in ('ecg', 'cxr'):
    train = [r for r in rows if r['split'] == 'train' and r['modality'] == modality]
    if len({r['patient_id'] for r in train}) < 2:
        raise ValueError(f'{modality}: need at least two training patients for own-patient exclusion.')
    print(f'{modality}: {len(train)} training reference records')

# Flush progress to the notebook and the log throughout the job.
# Restart this cell after a job interruption to reuse completed query batches.
import os
env = os.environ.copy()
workers = env.get('SLURM_CPUS_PER_TASK', '1')
for name in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'):
    env[name] = workers
log_path = OUT / 'build_banks.log'
# Release large duplicate input objects before starting the child process.
del rows, targets, train
with log_path.open('a', encoding='utf-8') as log:
    log.write(f'\nStarting/resuming bank build. Python: {sys.executable}\n')
    log.flush()
    process = subprocess.Popen(
        [sys.executable, '-u', str(script.resolve()), str(OUT.resolve())],
        cwd=str(ROOT.resolve()), env=env, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
        returncode = process.wait()
    except BaseException:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
    finally:
        process.stdout.close()
    log.write(f'Exit code: {returncode}\n')
if returncode != 0:
    raise RuntimeError(
        f'Bank construction failed (exit {returncode}). '
        f'The actual error is printed above and saved to {log_path.resolve()}. '
        'Completed batches are retained; rerun notebook 02 with unchanged inputs. '
        'Do not continue to notebook 03 until READY.'
    )
print(load(OUT / 'bank_audit.json'))
print('READY: bank construction completed. Diagnostic log:', log_path.resolve())
